<a href="https://colab.research.google.com/github/DanielYaari28/kaggle-philharmonia/blob/main/02_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

RAW = '/content/drive/MyDrive/kaggle-philharmonia/data/raw'
train   = pd.read_csv(f'{RAW}/train.csv')
test    = pd.read_csv(f'{RAW}/test.csv')
account = pd.read_csv(f'{RAW}/account.csv', encoding='latin-1')
subs    = pd.read_csv(f'{RAW}/subscriptions.csv')
tickets = pd.read_csv(f'{RAW}/tickets_all.csv')

Mounted at /content/drive


In [ ]:
# changing seasons to a singular value for easier feature engineering
subs['start'] = subs['season'].str[:4].astype(int)
subs[['season', 'start']].drop_duplicates().sort_values('start').tail()

,season,start
0,2009-2010,2009
18,2010-2011,2010
47,2011-2012,2011
7,2012-2013,2012
24,2013-2014,2013


In [ ]:
# investigate most recent season
g = subs.groupby('account.id')

sub_1314 = g['start'].apply(lambda x: int((x == 2013).any()))
sub_1314.head(), sub_1314.value_counts()

(account.id
 001i000000LhyPD    1
 001i000000LhyPE    1
 001i000000LhyPF    1
 001i000000LhyPG    1
 001i000000LhyPH    1
 Name: start, dtype: int64,
 start
 0    5285
 1     982
 Name: count, dtype: int64)

In [ ]:
n_sub_seasons  = g['start'].nunique()   # how many different seasons they subscribed
last_sub_start = g['start'].max()       # the most recent season they subscribed

sub_feats = pd.DataFrame({
    'sub_1314': sub_1314,
    'n_sub_seasons': n_sub_seasons,
    'last_sub_start': last_sub_start,
})
sub_feats.head()

,sub_1314,n_sub_seasons,last_sub_start
account.id,,,
001i000000LhyPD,1,12,2013
001i000000LhyPE,1,6,2013
001i000000LhyPF,1,19,2013
001i000000LhyPG,1,16,2013
001i000000LhyPH,1,11,2013


In [ ]:
feats = account[['account.id']].merge(sub_feats, left_on='account.id', right_index=True, how='left')
print(feats.shape)
print(feats.isna().sum())

(19833, 4)
account.id            0
sub_1314          13566
n_sub_seasons     13566
last_sub_start    13566
dtype: int64


In [ ]:
feats['ever_subscribed'] = feats['n_sub_seasons'].notna().astype(int)
feats['sub_1314']        = feats['sub_1314'].fillna(0).astype(int)
feats['n_sub_seasons']   = feats['n_sub_seasons'].fillna(0)
feats['seasons_since_last_sub'] = (2014 - feats['last_sub_start']).fillna(30)
feats = feats.drop(columns='last_sub_start')

print(feats.isna().sum())
feats.describe()

account.id                0
sub_1314                  0
n_sub_seasons             0
ever_subscribed           0
seasons_since_last_sub    0
dtype: int64


,sub_1314,n_sub_seasons,ever_subscribed,seasons_since_last_sub
count,19833.000000,19833.000000,19833.000000,19833.000000
mean,0.049513,1.443402,0.315989,24.213836
std,0.216943,3.710945,0.464920,9.327509
min,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,18.000000
50%,0.000000,0.000000,0.000000,30.000000
75%,0.000000,1.000000,1.000000,30.000000
max,1.000000,21.000000,1.000000,30.000000


In [ ]:
don_2013 = account.set_index('account.id')['amount.donated.2013']

feats['donor_2013']       = (feats['account.id'].map(don_2013) > 0).astype(int)
feats['log_donated_2013'] = np.log1p(feats['account.id'].map(don_2013))

feats[['donor_2013', 'log_donated_2013']].describe()

,donor_2013,log_donated_2013
count,19833.000000,19833.000000
mean,0.038169,0.217315
std,0.191608,1.136361
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,1.000000,12.261512


In [ ]:
feats['subscriber_x_donor'] = feats['ever_subscribed'] * feats['donor_2013']
pd.crosstab(feats['ever_subscribed'], feats['donor_2013'])

donor_2013,0,1
ever_subscribed,,
0,13440,126
1,5636,631


In [ ]:
feats['n_tickets'] = feats['account.id'].map(tickets.groupby('account.id').size()).fillna(0)
feats['n_tickets'].value_counts().head()

,count
n_tickets,
0.0,17784
1.0,1597
2.0,298
3.0,74
4.0,43


In [ ]:
FEATURES = [c for c in feats.columns if c != 'account.id']

X = train.merge(feats, on='account.id', how='left')[FEATURES]
y = train['label']
X_test = test.merge(feats, left_on='ID', right_on='account.id', how='left')[FEATURES]

print(X.shape, X_test.shape)
print("missing in X:", X.isna().sum().sum(), "| missing in X_test:", X_test.isna().sum().sum())
print(FEATURES)

(6941, 8) (2975, 8)
missing in X: 0 | missing in X_test: 0
['sub_1314', 'n_sub_seasons', 'ever_subscribed', 'seasons_since_last_sub', 'donor_2013', 'log_donated_2013', 'subscriber_x_donor', 'n_tickets']


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

oof = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
print(f"CV AUROC: {roc_auc_score(y, oof):.4f}")

CV AUROC: 0.9608


In [ ]:
import os
out_dir = '/content/drive/MyDrive/kaggle-philharmonia/submissions'
os.makedirs(out_dir, exist_ok=True)   # creates the folder; does nothing if it already exists

submission.to_csv(f'{out_dir}/01_baseline_logreg.csv', index=False)
print("saved:", os.listdir(out_dir))

saved: ['01_baseline_logreg.csv']


In [ ]:
model.fit(X, y)
test_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({'ID': test['ID'], 'Predicted': test_pred})
print(submission.shape)
print(submission.head())
print(submission['Predicted'].describe())

submission.to_csv('/content/drive/MyDrive/kaggle-philharmonia/submissions/01_baseline_logreg.csv', index=False)

(2975, 2)
                ID  Predicted
0  001i000000NuQ6Y   0.030040
1  001i000000NuQXz   0.028541
2  001i000000NuRDC   0.009214
3  001i000000NuQeg   0.005823
4  001i000000NuOQc   0.037532
count    2975.000000
mean        0.048709
std         0.181538
min         0.000369
25%         0.005823
50%         0.005823
75%         0.005823
max         0.984932
Name: Predicted, dtype: float64


In [ ]:
import os
out_dir = '/content/drive/MyDrive/kaggle-philharmonia/submissions'
os.makedirs(out_dir, exist_ok=True)

submission.to_csv(f'{out_dir}/01_baseline_logreg.csv', index=False)
print("saved:", os.listdir(out_dir))

saved: ['01_baseline_logreg.csv']
